# Day 6 — Blank-file pandas test

**3 tasks · 15 minutes each · 45 minutes total**

> The goal is not to finish fast. The goal is to prove you can start from an **empty cell** and
> write the analysis yourself, without copying from the guide.

### Rules

1. Close `guide.ipynb` and `eda_report.ipynb`.
2. No AI assistant, no copy-paste from old notebooks. The official pandas documentation is allowed
   (looking up a function name is normal; copying a whole solution is not).
3. **15 minutes per task.** When the timer rings, stop and grade, even if unfinished.
4. Write your answer in the empty cell under each task, then write the plain-language conclusion in
   the markdown cell below it.
5. Only open `blank_file_test_solutions.ipynb` after all three tasks.

### Setup (already written for you)

Run the next cell once. It loads the data into a DataFrame called `raw`.

### How to grade yourself

For each task, count it as **pass** only if your code runs **and** your numbers match the
self-check (open the `<details>` block after finishing). 2 out of 3 = ready. Below 2 = repeat the
test tomorrow with the guide closed.

In [ ]:

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)

DATA = next(p for p in [Path("data/telecom_churn.csv"), Path("../d5/telecom_churn.csv")] if p.exists())
raw = pd.read_csv(DATA)
print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")


---

## Task 1 — Data health check (15 min)

Write code that answers, in one or two cells:

1. How many **rows and columns** does the dataset have?
2. Build a **missing-values table** with the number missing, the percentage, and the dtype for every
   affected column — sorted from worst to least.
3. How many **duplicate rows** are there? Is `customer_id` unique?
4. What would `raw.dropna()` do to the dataset (how many rows would remain)?

Then write a **3-sentence verdict** in the markdown cell below: what is healthy, what is worrying,
and what you would do about it.

*Hint: `isna().sum()`, `duplicated()`, `is_unique`, `dropna().shape`.*

**Your verdict (3 sentences):**

1.
2.
3.

<details>
<summary><b>Self-check — open only after the timer</b></summary>

* Shape: **3,500 rows × 23 columns**.
* Missing: `tech_support` 60 (1.71%), `total_charges` 45 (1.29%),
  `customer_service_calls` 35 (1.00%), `cancellation_request` 1,553 (44.37%).
* Duplicates: **0**; `customer_id` is unique (**True**).
* `raw.dropna()` leaves **1,866 rows** — 1,634 rows (46.7%) would disappear.
* A good verdict names the 44%-missing column as *informative*, not random, and refuses to drop
  rows / impute blindly.

</details>

---

## Task 2 — Segment churn (15 min)

Using `raw`:

1. Compute **customers** and **churn rate** for each `contract` type, sorted from highest to lowest
   churn. Show the rate as a percentage.
2. Create a `tenure_band` column with `pd.cut` using the bins `[0, 12, 24, 36, 48, 60, 72]` and the
   labels `["1-12", "13-24", "25-36", "37-48", "49-60", "61-72"]`, then compute churn rate per band.
3. Find the **contract + tenure** segment with at least **100 customers** and the highest churn rate.
   Print it.
4. Draw one bar chart of churn rate by contract type.

*Hint: `groupby(...).agg(n="size", rate="mean")`, `pd.cut`, `bar_label`.*

<details>
<summary><b>Self-check — open only after the timer</b></summary>

* Contract: month-to-month **70.6%** (1,937 customers), one year **49.0%** (764),
  two year **25.7%** (799).
* Tenure bands: **69.8%**, 61.6%, 46.5%, 37.8%, 35.8%, **18.1%**.
* Highest segment with n ≥ 100: **month-to-month + 1–12 months = 83.7% (789 customers)**.
* The chart must have a title, a labelled y-axis and percentage labels.

</details>

---

## Task 3 — Leakage audit (15 min)

A colleague claims: *"I trained a model with all the columns and got 99.9% accuracy. We're done!"*
Your job is to check whether that is real.

1. For each of these columns, print a table of churn rate by value (fill missing with `"(missing)"`):
   `cancellation_request`, `last_interaction_status`, `churn_risk_post_exit_code`,
   `satisfaction_score`.
2. Decide which columns are **leakage** (recorded at or after the exit). Write the list in a Python
   variable called `leakage_cols`.
3. Create a clean DataFrame called `clean` that drops `customer_id` and every column in
   `leakage_cols`. Print its shape.
4. Draw one chart that makes the leakage obvious, and write one plain-language sentence about why
   99.9% accuracy is a red flag.

*Hint: `pd.crosstab(..., normalize="columns")` or `groupby(...)["churn"].agg(...)`, then `drop`.*

**Your sentence about the 99.9% claim:**

<details>
<summary><b>Self-check — open only after the timer</b></summary>

* All four columns are leakage:

  | column | pattern |
  |---|---|
  | `cancellation_request` | `"Recorded"` = 100% churn; missing = 0% churn |
  | `last_interaction_status` | "Account Closed" / "Retention Declined" / "Final Invoice Dispatched" = 100% churn |
  | `churn_risk_post_exit_code` | `999` = churn, `100` = stayed |
  | `satisfaction_score` | 100% churn at score 1, 0% at score 5 (survey taken at exit) |

* `clean.shape` must be **(3500, 18)**.
* A good sentence: *"99.9% accuracy means the model found a column that records the outcome,
  not a pattern that predicts it — so it will fail on any customer who has not left yet."*

</details>

---

## Score sheet

| Task | Finished in time? | Numbers match? | Pass (Y/N) |
|---|---|---|---|
| 1 — data health check | | | |
| 2 — segment churn | | | |
| 3 — leakage audit | | | |

**Score: ___ / 3**

### Reflection (write after grading)

* Which task was slowest, and why? (syntax, forgetting a function, or not knowing the next step?)
* Which single pandas pattern do you want to memorise before tomorrow?
* If you had 10 more minutes, what would you fix?

**Next step:** redo only the tasks you failed, two days later, with the guide closed.